# Music Source Separation - Model Evaluation

This notebook evaluates a trained model on test data and visualizes results.

In [ ]:
import sys
sys.path.append('..')

import torch
import torchaudio
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import IPython.display as ipd
from tqdm import tqdm

from src.model import UNet, MaskBasedUNet
from src.dataset import MusicSeparationDataset
from src.metrics import compute_separation_metrics, MetricsTracker
from src.utils import load_config, load_audio, stft, istft, magnitude_phase, save_audio
from torch.utils.data import DataLoader

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Load Configuration and Model

In [ ]:
config = load_config('../config.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Checkpoint path - modify this to your trained model
checkpoint_path = '../checkpoints/best_model.pth'

print(f"\nLoading checkpoint from: {checkpoint_path}")

In [ ]:
# Create model
model = MaskBasedUNet(
    in_channels=config['model']['in_channels'],
    out_channels=config['model']['out_channels'],
    num_stems=config['model']['num_stems'],
    base_channels=config['model']['base_channels'],
    depth=config['model']['depth'],
    kernel_size=config['model']['kernel_size']
).to(device)

# Load checkpoint
if Path(checkpoint_path).exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✓ Loaded checkpoint from epoch {checkpoint['epoch']}")
    print(f"  Best validation loss: {checkpoint['best_val_loss']:.4f}")
else:
    print(f"⚠️ Checkpoint not found at {checkpoint_path}")
    print("  Using untrained model for demonstration")

model.eval()
print("\nModel ready for evaluation!")

## 2. Load Test Dataset

In [ ]:
test_dataset = MusicSeparationDataset(
    raw_dir=config['data']['raw_dir'],
    stems_dir=config['data']['stems_dir'],
    stems=config['data']['stems'],
    sample_rate=config['data']['sample_rate'],
    segment_length=config['data']['segment_length'],
    n_fft=config['data']['n_fft'],
    hop_length=config['data']['hop_length'],
    win_length=config['data']['win_length'],
    train=False
)

print(f"Test dataset size: {len(test_dataset)}")

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0
)

## 3. Evaluate on Test Set

In [ ]:
print("Evaluating model on test set...\n")

metrics_tracker = MetricsTracker(config['data']['stems'])
all_results = []

with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(test_loader, desc="Processing")):
        # Move to device
        mix_mag = batch['mix_magnitude'].to(device)
        mix_phase = batch['mix_phase'].to(device)
        song_name = batch['song_name'][0]
        
        # Get predictions
        output = model(mix_mag)
        
        # Reshape to separate stems
        batch_size, _, freq, time = output.shape
        output = output.view(
            batch_size,
            config['model']['num_stems'],
            config['model']['out_channels'],
            freq,
            time
        )
        
        # Reconstruct audio
        pred_audios = {}
        ref_audios = {}
        
        for i, stem in enumerate(config['data']['stems']):
            # Predicted stem
            pred_mag = output[:, i, :, :, :]
            pred_complex = pred_mag * torch.exp(1j * mix_phase)
            pred_audio = istft(
                pred_complex,
                n_fft=config['data']['n_fft'],
                hop_length=config['data']['hop_length'],
                win_length=config['data']['win_length']
            )
            
            # Reference stem
            ref_mag = batch['stem_magnitudes'][stem].to(device)
            ref_complex = ref_mag * torch.exp(1j * mix_phase)
            ref_audio = istft(
                ref_complex,
                n_fft=config['data']['n_fft'],
                hop_length=config['data']['hop_length'],
                win_length=config['data']['win_length']
            )
            
            pred_audios[stem] = pred_audio
            ref_audios[stem] = ref_audio
        
        # Compute metrics
        batch_metrics = compute_separation_metrics(
            pred_audios,
            ref_audios,
            config['data']['stems']
        )
        
        metrics_tracker.update(batch_metrics)
        all_results.append({
            'song_name': song_name,
            'metrics': batch_metrics
        })

print("\n✓ Evaluation completed!")

## 4. Display Overall Metrics

In [ ]:
avg_metrics = metrics_tracker.compute_average()

print("=" * 60)
print("OVERALL METRICS")
print("=" * 60)
print(metrics_tracker.get_summary_string())
print("=" * 60)

## 5. Visualize Metrics

In [ ]:
# Plot metrics per stem
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_names = ['sdr', 'sir', 'sar']
stems = config['data']['stems']

for idx, metric_name in enumerate(metrics_names):
    values = [avg_metrics[stem][metric_name] for stem in stems if metric_name in avg_metrics[stem]]
    
    axes[idx].bar(stems, values, color='skyblue', edgecolor='navy', alpha=0.7)
    axes[idx].set_title(f'{metric_name.upper()} by Stem', fontsize=14, fontweight='bold')
    axes[idx].set_ylabel('dB', fontsize=12)
    axes[idx].grid(True, alpha=0.3, axis='y')
    axes[idx].axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.5)
    
    # Add value labels on bars
    for i, v in enumerate(values):
        axes[idx].text(i, v + 0.5, f'{v:.2f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 6. Per-Song Metrics

In [ ]:
print("Per-Song Metrics:\n")

for result in all_results:
    print(f"Song: {result['song_name']}")
    for stem in config['data']['stems']:
        if stem in result['metrics']:
            metrics = result['metrics'][stem]
            print(f"  {stem}:")
            for metric_name, value in metrics.items():
                print(f"    {metric_name.upper()}: {value:.2f} dB")
    print()

## 7. Visualize Separation Results

In [ ]:
# Select a sample for detailed visualization
sample_idx = 0
sample = test_dataset[sample_idx]

print(f"Visualizing sample: {sample['song_name']}")

# Get predictions
mix_mag = sample['mix_magnitude'].unsqueeze(0).to(device)
mix_phase = sample['mix_phase'].unsqueeze(0).to(device)

with torch.no_grad():
    output = model(mix_mag)
    output = output.view(1, config['model']['num_stems'], config['model']['out_channels'], -1, output.shape[-1])

In [ ]:
# Plot spectrograms
num_stems = len(config['data']['stems'])
fig, axes = plt.subplots(num_stems + 1, 3, figsize=(18, 4 * (num_stems + 1)))

# Plot mix
mix_db = 20 * torch.log10(mix_mag[0, 0] + 1e-8).cpu().numpy()
axes[0, 0].imshow(mix_db, aspect='auto', origin='lower', cmap='viridis', vmin=-80, vmax=0)
axes[0, 0].set_title('Input Mix', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Frequency Bins')
axes[0, 1].axis('off')
axes[0, 2].axis('off')

# Plot each stem
for i, stem in enumerate(config['data']['stems']):
    # Reference
    ref_mag = sample['stem_magnitudes'][stem]
    ref_db = 20 * torch.log10(ref_mag[0] + 1e-8).numpy()
    
    axes[i+1, 0].imshow(ref_db, aspect='auto', origin='lower', cmap='viridis', vmin=-80, vmax=0)
    axes[i+1, 0].set_title(f'{stem.capitalize()} - Reference', fontsize=10)
    axes[i+1, 0].set_ylabel('Frequency')
    
    # Predicted
    pred_mag = output[0, i, 0]
    pred_db = 20 * torch.log10(pred_mag + 1e-8).cpu().numpy()
    
    axes[i+1, 1].imshow(pred_db, aspect='auto', origin='lower', cmap='viridis', vmin=-80, vmax=0)
    axes[i+1, 1].set_title(f'{stem.capitalize()} - Predicted', fontsize=10)
    
    # Difference
    diff = np.abs(ref_db - pred_db)
    im = axes[i+1, 2].imshow(diff, aspect='auto', origin='lower', cmap='hot', vmin=0, vmax=20)
    axes[i+1, 2].set_title(f'{stem.capitalize()} - Difference', fontsize=10)
    plt.colorbar(im, ax=axes[i+1, 2])
    
    if i == num_stems - 1:
        for j in range(3):
            axes[i+1, j].set_xlabel('Time Frames')

plt.tight_layout()
plt.show()

## 8. Listen to Separated Audio

In [ ]:
print(f"Audio samples for: {sample['song_name']}\n")

# Original mix
print("Original Mix:")
ipd.display(ipd.Audio(sample['mix_audio'].numpy(), rate=config['data']['sample_rate']))

# Separated stems
for i, stem in enumerate(config['data']['stems']):
    # Reconstruct audio
    pred_mag = output[0, i, :, :, :]
    pred_complex = pred_mag * torch.exp(1j * mix_phase)
    pred_audio = istft(
        pred_complex,
        n_fft=config['data']['n_fft'],
        hop_length=config['data']['hop_length'],
        win_length=config['data']['win_length']
    )
    
    print(f"\n{stem.capitalize()} - Predicted:")
    ipd.display(ipd.Audio(pred_audio.cpu().numpy(), rate=config['data']['sample_rate']))
    
    # Reference
    ref_mag = sample['stem_magnitudes'][stem].unsqueeze(0)
    ref_complex = ref_mag * torch.exp(1j * mix_phase.cpu())
    ref_audio = istft(
        ref_complex,
        n_fft=config['data']['n_fft'],
        hop_length=config['data']['hop_length'],
        win_length=config['data']['win_length']
    )
    
    print(f"{stem.capitalize()} - Reference:")
    ipd.display(ipd.Audio(ref_audio.cpu().numpy(), rate=config['data']['sample_rate']))

## 9. Export Separated Audio

In [ ]:
# Create output directory
output_dir = Path('../outputs/separated')
output_dir.mkdir(parents=True, exist_ok=True)

song_output_dir = output_dir / sample['song_name']
song_output_dir.mkdir(exist_ok=True)

print(f"Exporting separated audio to: {song_output_dir}\n")

# Save separated stems
for i, stem in enumerate(config['data']['stems']):
    # Reconstruct audio
    pred_mag = output[0, i, :, :, :]
    pred_complex = pred_mag * torch.exp(1j * mix_phase)
    pred_audio = istft(
        pred_complex,
        n_fft=config['data']['n_fft'],
        hop_length=config['data']['hop_length'],
        win_length=config['data']['win_length']
    )
    
    # Save
    output_file = song_output_dir / f"{stem}.wav"
    save_audio(str(output_file), pred_audio.cpu(), config['data']['sample_rate'])
    print(f"✓ Saved: {output_file}")

print("\n✓ Export completed!")

## 10. Metrics Summary Table

In [ ]:
import pandas as pd

# Create summary DataFrame
summary_data = []
for stem in config['data']['stems'] + ['average']:
    if stem in avg_metrics:
        row = {'Stem': stem.capitalize()}
        for metric in ['sdr', 'sir', 'sar']:
            if metric in avg_metrics[stem]:
                row[metric.upper()] = f"{avg_metrics[stem][metric]:.2f}"
        summary_data.append(row)

df = pd.DataFrame(summary_data)
print("\nMetrics Summary Table:")
print(df.to_string(index=False))

## Summary

✅ Model evaluation on test set
✅ Metrics computation (SDR, SIR, SAR)
✅ Visualization of separated spectrograms
✅ Audio playback comparison
✅ Export separated stems

**Key Metrics:**
- **SDR (Signal-to-Distortion Ratio)**: Overall quality (higher is better)
- **SIR (Signal-to-Interference Ratio)**: Suppression of other sources (higher is better)
- **SAR (Signal-to-Artifacts Ratio)**: Artificial distortions (higher is better)

**Typical Good Values:**
- SDR > 6 dB: Acceptable separation
- SDR > 10 dB: Good separation
- SDR > 15 dB: Excellent separation